[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-8_executed.ipynb)

---


# 93. The Inventory-Routing Problem (IRP)
## Tier 8: A Value-Aligned and Ethical Framework

### Purpose and scope

This notebook demonstrates a staged heuristic for considering ethical principles and stakeholder interests in a simplified inventory-routing setting. It creates a small instance, generates an initial customer-to-vehicle assignment, evaluates principle-compliance scores, applies limited route and delivery adjustments, calculates stakeholder utilities, and reports the resulting metrics.

The example is an educational simulation, not a complete IRP solver and not a formal implementation of Constitutional AI. In particular, it does not optimize inventory over time, enforce vehicle capacities, model uncertain demand, or solve a mathematical program to optimality. Each planning period is evaluated independently using the same static instance. The carbon and service-equity measures are proxies; the labor, resource-utilization, and community-impact constraints do not have enough operational data in this example to be evaluated.

### Modeling assumptions

- The depot and customers have two-dimensional coordinates, and travel distance is Euclidean.
- Customers are assigned to vehicles in their input order. Within a route, a nearest-next-customer rule orders the assigned customers when an environmental adjustment is triggered.
- The initial delivery to each assigned customer is twice that customer's mean demand. Delivery quantities are not constrained by vehicle capacity or inventory bounds.
- A route incurs its vehicle's fixed cost when it serves at least one customer, plus its distance multiplied by that vehicle's variable cost.
- Emissions are estimated as 0.5 kg CO₂ per distance unit. This is a modeling proxy, not a measured emissions factor.
- The six periods are repeated static evaluations; they do not represent inventory carryover or six distinct demand realizations.
- Ethical weights, priorities, thresholds, and simplified utility metrics are illustrative model inputs. Scores are not external audits or evidence of real-world ethical compliance.

### Method and mathematical representation

The method follows five stages for each period: generate an initial solution; evaluate principle compliance; apply the prescribed ethical adjustments and violation penalties; calculate stakeholder utilities; and finalize and report the solution. The route-order adjustment keeps every assigned customer in its route. It chooses the next customer with the smallest Euclidean distance from the current location, beginning at the depot, and then returns the route to the depot for distance accounting.

Let \(C\) be the set of customers and \(K\) the set of vehicles. A route \(R_k\) is the ordered list of customers assigned to vehicle \(k\), and \(q_i\) is the delivery to customer \(i\). For a route with coordinates \(p_0\) (the depot), \(p_1,\ldots,p_m\) (the customers in route order), and \(p_{m+1}=p_0\), its length is

$$
L_k = \sum_{j=0}^{m} \lVert p_{j+1}-p_j \rVert_2.
$$

The distance-based cost used in the code is

$$
C_{\mathrm{total}} = \sum_{k\in K}\left(f_k\,\mathbf{1}[R_k\ne\varnothing] + v_k L_k\right) + P,
$$

where \(f_k\) and \(v_k\) are the vehicle's fixed and variable costs, and \(P\) is the sum of penalties for the implemented carbon and service-equity violations. The indicator is 1 when vehicle \(k\) serves at least one customer and 0 otherwise.

For deliveries with positive mean \(\bar q\), the service-equity proxy is

$$
E(q)=\max\left(0,\,1-\frac{\operatorname{std}(q)}{\bar q}\right).
$$

The principle-compliance and stakeholder-utility scores are bounded, illustrative scores defined by their corresponding code methods; they are not decision variables in a globally optimized objective. The environmental adjustment is applied when the initial Environmental Sustainability score is below 0.8, and delivery balancing is applied when the initial Social Equity score is below 0.8. A violation penalty is then calculated for the adjusted solution. Compliance is evaluated again after these adjustments so that reported scores describe the final routes and deliveries.

In the implementation, `_generate_initial_solution` constructs routes and deliveries, `_calculate_total_distance` evaluates route lengths, `_evaluate_ethical_compliance` and `_calculate_principle_compliance` calculate principle scores, and `_optimize_ethically` applies adjustments and penalties. The later methods calculate stakeholder utilities, create the result record, and aggregate period-level metrics. The procedure is heuristic and does not guarantee a globally optimal route or a feasible solution to a complete IRP.

### What to examine

Interpret the compliance scores alongside their definitions and thresholds. A score is not proof that an ethical obligation has been met: some measures are simplified proxies, and several listed constraints cannot be assessed with the available data. Stakeholder utility scores are weighted summaries of selected modeled metrics, not direct measures of stakeholder well-being. The visualizations summarize the same simulated results; they do not provide independent validation.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class ConstitutionalPrinciple:
    """A named principle with a weight and a measurable target."""

    name: str
    description: str
    weight: float
    constraint_type: str
    metric_function: str
    target_value: float


@dataclass
class Stakeholder:
    """A stakeholder and the weights assigned to modeled utility metrics."""

    name: str
    stakeholder_type: str
    utility_function: Dict[str, float]
    priority: float
    impact_areas: List[str]


@dataclass
class EthicalConstraint:
    """A constraint specification and its penalty or reporting parameters."""

    name: str
    constraint_type: str
    limit_value: float
    penalty_weight: float
    measurement_method: str
    compliance_threshold: float


@dataclass
class ValueAlignedSolution:
    """Metrics and decisions recorded for one simulated period."""

    routes: Dict[int, List[int]]
    deliveries: Dict[int, float]
    total_cost: float
    service_level: float
    ethical_compliance: Dict[str, float]
    stakeholder_utility: Dict[str, float]
    environmental_impact: float
    social_impact: float
    economic_impact: float
    overall_ethical_score: float


class ConstitutionalAI:
    """Apply the notebook's staged heuristic to a simplified IRP instance."""

    EMISSIONS_PER_DISTANCE_UNIT = 0.5

    def __init__(self, depot, customers, vehicles, num_periods):
        if not customers:
            raise ValueError("At least one customer is required.")
        if not vehicles:
            raise ValueError("At least one vehicle is required.")
        if num_periods <= 0:
            raise ValueError("num_periods must be a positive integer.")

        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.customer_by_id = {customer.id: customer for customer in customers}
        self.vehicle_by_id = {vehicle.id: vehicle for vehicle in vehicles}

        if len(self.customer_by_id) != len(customers):
            raise ValueError("Customer IDs must be unique.")
        if len(self.vehicle_by_id) != len(vehicles):
            raise ValueError("Vehicle IDs must be unique.")

        self.principles = self._initialize_constitutional_principles()
        self.stakeholders = self._initialize_stakeholders()
        self.ethical_constraints = self._initialize_ethical_constraints()
        self.solution_history = []
        self.compliance_history = []
        self.utility_history = []

        print(
            f"Initialized the framework with {len(self.principles)} principles "
            f"and {len(self.stakeholders)} stakeholders."
        )

    def _initialize_constitutional_principles(self) -> List[ConstitutionalPrinciple]:
        """Define the principles and score targets used in this example."""
        return [
            ConstitutionalPrinciple(
                name="Environmental Sustainability",
                description="Reduce estimated carbon emissions.",
                weight=0.25,
                constraint_type="hard",
                metric_function="carbon_emission",
                target_value=100.0,
            ),
            ConstitutionalPrinciple(
                name="Social Equity",
                description="Promote an even distribution of deliveries.",
                weight=0.20,
                constraint_type="soft",
                metric_function="service_equity",
                target_value=0.90,
            ),
            ConstitutionalPrinciple(
                name="Economic Viability",
                description="Keep cost per customer within a normalized range.",
                weight=0.20,
                constraint_type="soft",
                metric_function="cost_efficiency",
                target_value=0.80,
            ),
            ConstitutionalPrinciple(
                name="Transparency",
                description="Make the calculation process inspectable.",
                weight=0.15,
                constraint_type="guideline",
                metric_function="decision_transparency",
                target_value=0.85,
            ),
            ConstitutionalPrinciple(
                name="Accountability",
                description="Record modeled impacts and decisions.",
                weight=0.20,
                constraint_type="hard",
                metric_function="impact_accountability",
                target_value=0.90,
            ),
        ]

    def _initialize_stakeholders(self) -> List[Stakeholder]:
        """Define illustrative stakeholder metric weights and priorities."""
        return [
            Stakeholder(
                name="Customers",
                stakeholder_type="customer",
                utility_function={
                    "service_level": 0.40,
                    "reliability": 0.30,
                    "cost": 0.20,
                    "environmental_impact": 0.10,
                },
                priority=0.90,
                impact_areas=["service quality", "pricing", "environmental footprint"],
            ),
            Stakeholder(
                name="Employees",
                stakeholder_type="employee",
                utility_function={
                    "job_security": 0.35,
                    "working_conditions": 0.30,
                    "fair_wages": 0.25,
                    "work_life_balance": 0.10,
                },
                priority=0.85,
                impact_areas=["employment conditions", "workload", "safety"],
            ),
            Stakeholder(
                name="Community",
                stakeholder_type="community",
                utility_function={
                    "environmental_quality": 0.30,
                    "economic_development": 0.25,
                    "traffic_congestion": 0.25,
                    "noise_pollution": 0.20,
                },
                priority=0.80,
                impact_areas=["environmental impact", "local economy", "quality of life"],
            ),
            Stakeholder(
                name="Environment",
                stakeholder_type="environment",
                utility_function={
                    "carbon_emissions": 0.40,
                    "resource_consumption": 0.30,
                    "waste_generation": 0.20,
                    "biodiversity_impact": 0.10,
                },
                priority=0.95,
                impact_areas=["carbon footprint", "resource use", "pollution"],
            ),
            Stakeholder(
                name="Shareholders",
                stakeholder_type="shareholder",
                utility_function={
                    "profitability": 0.40,
                    "risk_management": 0.25,
                    "growth_potential": 0.20,
                    "reputation": 0.15,
                },
                priority=0.75,
                impact_areas=["financial performance", "risk exposure", "market position"],
            ),
        ]

    def _initialize_ethical_constraints(self) -> List[EthicalConstraint]:
        """Store constraint targets, noting that not all are measurable here."""
        return [
            EthicalConstraint(
                name="Carbon Emission Limit",
                constraint_type="environmental",
                limit_value=150.0,
                penalty_weight=0.30,
                measurement_method="distance-based emissions proxy",
                compliance_threshold=0.90,
            ),
            EthicalConstraint(
                name="Service Equity Requirement",
                constraint_type="social",
                limit_value=0.85,
                penalty_weight=0.25,
                measurement_method="delivery-distribution proxy",
                compliance_threshold=0.80,
            ),
            EthicalConstraint(
                name="Fair Labor Standards",
                constraint_type="social",
                limit_value=8.0,
                penalty_weight=0.20,
                measurement_method="working-time data not provided",
                compliance_threshold=0.95,
            ),
            EthicalConstraint(
                name="Resource Utilization Limit",
                constraint_type="environmental",
                limit_value=0.80,
                penalty_weight=0.15,
                measurement_method="resource-consumption data not provided",
                compliance_threshold=0.85,
            ),
            EthicalConstraint(
                name="Community Impact Limit",
                constraint_type="social",
                limit_value=0.70,
                penalty_weight=0.10,
                measurement_method="community-impact data not provided",
                compliance_threshold=0.75,
            ),
        ]

    def optimize_with_ethical_constraints(self, period: int) -> ValueAlignedSolution:
        """Run the five stages of the heuristic for one period."""
        print(f"\n=== ETHICAL EVALUATION: Period {period + 1} ===")

        print("Stage 1: Generating an initial solution...")
        initial_solution = self._generate_initial_solution(period)

        print("Stage 2: Evaluating initial principle compliance...")
        initial_compliance = self._evaluate_ethical_compliance(initial_solution)

        print("Stage 3: Applying ethical adjustments and penalties...")
        adjusted_solution = self._optimize_ethically(
            initial_solution, initial_compliance
        )
        final_compliance = self._evaluate_ethical_compliance(adjusted_solution)

        print("Stage 4: Calculating stakeholder utilities...")
        stakeholder_utilities = self._calculate_stakeholder_utilities(adjusted_solution)

        print("Stage 5: Finalizing and reporting the solution...")
        final_solution = self._finalize_solution(
            adjusted_solution, final_compliance, stakeholder_utilities
        )

        self.solution_history.append(final_solution)
        self.compliance_history.append(final_compliance)
        self.utility_history.append(stakeholder_utilities)
        self._report_ethical_results(
            final_solution, final_compliance, stakeholder_utilities
        )
        return final_solution

    def _generate_initial_solution(self, period: int) -> Dict[str, Any]:
        """Assign customers in input order and set illustrative deliveries."""
        del period  # The example uses the same static data in every period.
        routes = {vehicle.id: [] for vehicle in self.vehicles}
        deliveries = {}
        customers_per_vehicle = len(self.customers) // len(self.vehicles)

        for vehicle_index, vehicle in enumerate(self.vehicles):
            start_index = vehicle_index * customers_per_vehicle
            end_index = start_index + customers_per_vehicle
            if vehicle_index == len(self.vehicles) - 1:
                end_index = len(self.customers)

            assigned_customers = self.customers[start_index:end_index]
            routes[vehicle.id] = [customer.id for customer in assigned_customers]
            for customer in assigned_customers:
                deliveries[customer.id] = customer.demand_mean * 2.0

        total_distance = self._calculate_total_distance(routes)
        total_cost = self._calculate_base_cost(routes, total_distance)
        return {
            "routes": routes,
            "deliveries": deliveries,
            "total_distance": total_distance,
            "total_cost": total_cost,
            "service_level": 0.85,
            "ethical_penalty": 0.0,
        }

    def _calculate_total_distance(self, routes: Dict[int, List[int]]) -> float:
        """Sum Euclidean travel distance, including each depot return leg."""
        total_distance = 0.0
        depot_position = (self.depot.x, self.depot.y)

        for customer_ids in routes.values():
            if not customer_ids:
                continue
            current_position = depot_position
            for customer_id in customer_ids:
                customer = self.customer_by_id[customer_id]
                next_position = (customer.x, customer.y)
                total_distance += float(
                    np.hypot(
                        next_position[0] - current_position[0],
                        next_position[1] - current_position[1],
                    )
                )
                current_position = next_position
            total_distance += float(
                np.hypot(
                    depot_position[0] - current_position[0],
                    depot_position[1] - current_position[1],
                )
            )
        return total_distance

    def _calculate_base_cost(
        self, routes: Dict[int, List[int]], total_distance: float
    ) -> float:
        """Calculate vehicle-specific fixed and distance-based operating costs."""
        if total_distance == 0:
            return 0.0
        distance_by_vehicle = {
            vehicle_id: self._calculate_route_distance(customer_ids)
            for vehicle_id, customer_ids in routes.items()
        }
        return sum(
            (self.vehicle_by_id[vehicle_id].fixed_cost if customer_ids else 0.0)
            + self.vehicle_by_id[vehicle_id].variable_cost
            * distance_by_vehicle[vehicle_id]
            for vehicle_id, customer_ids in routes.items()
        )

    def _calculate_route_distance(self, customer_ids: List[int]) -> float:
        """Calculate the depot-to-depot distance for one ordered route."""
        if not customer_ids:
            return 0.0
        positions = [(self.depot.x, self.depot.y)]
        positions.extend(
            (self.customer_by_id[customer_id].x, self.customer_by_id[customer_id].y)
            for customer_id in customer_ids
        )
        positions.append((self.depot.x, self.depot.y))
        return sum(
            float(np.hypot(end[0] - start[0], end[1] - start[1]))
            for start, end in zip(positions, positions[1:])
        )

    def _evaluate_ethical_compliance(
        self, solution: Dict[str, Any]
    ) -> Dict[str, float]:
        """Return one bounded compliance score for each modeled principle."""
        return {
            principle.name: self._calculate_principle_compliance(principle, solution)
            for principle in self.principles
        }

    def _calculate_principle_compliance(
        self,
        principle: ConstitutionalPrinciple,
        solution: Dict[str, Any],
    ) -> float:
        """Calculate the example's proxy score for one constitutional principle."""
        if principle.metric_function == "carbon_emission":
            emissions = (
                solution["total_distance"] * self.EMISSIONS_PER_DISTANCE_UNIT
            )
            compliance = 1.0 - (
                emissions - principle.target_value
            ) / principle.target_value
        elif principle.metric_function == "service_equity":
            equity = self._calculate_service_equity(solution["deliveries"])
            compliance = min(1.0, equity / principle.target_value)
        elif principle.metric_function == "cost_efficiency":
            cost_per_customer = solution["total_cost"] / len(self.customers)
            compliance = 1.0 - cost_per_customer / 500.0
        elif principle.metric_function == "decision_transparency":
            compliance = 0.80
        elif principle.metric_function == "impact_accountability":
            compliance = 0.90
        else:
            raise ValueError(
                f"Unsupported principle metric: {principle.metric_function}"
            )
        return float(np.clip(compliance, 0.0, 1.0))

    @staticmethod
    def _calculate_service_equity(deliveries: Dict[int, float]) -> float:
        """Measure delivery equality using one minus the coefficient of variation."""
        values = np.asarray(list(deliveries.values()), dtype=float)
        if values.size == 0:
            return 0.0
        mean_delivery = float(np.mean(values))
        if mean_delivery <= 0.0:
            return 0.0
        return float(np.clip(1.0 - np.std(values) / mean_delivery, 0.0, 1.0))

    def _optimize_ethically(
        self,
        solution: Dict[str, Any],
        compliance_scores: Dict[str, float],
    ) -> Dict[str, Any]:
        """Apply the example's conditional adjustments and final-solution penalties."""
        optimized_solution = {
            **solution,
            "routes": {
                vehicle_id: list(customer_ids)
                for vehicle_id, customer_ids in solution["routes"].items()
            },
            "deliveries": dict(solution["deliveries"]),
        }

        if compliance_scores["Environmental Sustainability"] < 0.8:
            optimized_solution = self._reduce_environmental_impact(
                optimized_solution
            )
        if compliance_scores["Social Equity"] < 0.8:
            optimized_solution = self._improve_service_equity(optimized_solution)

        total_distance = self._calculate_total_distance(optimized_solution["routes"])
        base_cost = self._calculate_base_cost(
            optimized_solution["routes"], total_distance
        )
        optimized_solution["total_distance"] = total_distance
        optimized_solution["ethical_penalty"] = sum(
            self._calculate_constraint_violation(constraint, optimized_solution)
            for constraint in self.ethical_constraints
        )
        optimized_solution["total_cost"] = (
            base_cost + optimized_solution["ethical_penalty"]
        )
        return optimized_solution

    def _calculate_constraint_violation(
        self,
        constraint: EthicalConstraint,
        solution: Dict[str, Any],
    ) -> float:
        """Return a weighted penalty for the two constraints with available data."""
        if constraint.name == "Carbon Emission Limit":
            emissions = (
                solution["total_distance"] * self.EMISSIONS_PER_DISTANCE_UNIT
            )
            violation = max(0.0, emissions - constraint.limit_value)
            return violation * constraint.penalty_weight * 10.0
        if constraint.name == "Service Equity Requirement":
            equity = self._calculate_service_equity(solution["deliveries"])
            violation = max(0.0, constraint.limit_value - equity)
            return violation * constraint.penalty_weight * 100.0
        # The remaining constraints require measurements absent from this instance.
        return 0.0

    def _reduce_environmental_impact(
        self, solution: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Reorder each assigned route by nearest next customer without dropping service."""
        optimized_solution = {
            **solution,
            "routes": {
                vehicle_id: list(customer_ids)
                for vehicle_id, customer_ids in solution["routes"].items()
            },
            "deliveries": dict(solution["deliveries"]),
        }

        for vehicle_id, customer_ids in optimized_solution["routes"].items():
            remaining = list(customer_ids)
            ordered = []
            current_position = (self.depot.x, self.depot.y)
            while remaining:
                next_customer_id = min(
                    remaining,
                    key=lambda customer_id: np.hypot(
                        self.customer_by_id[customer_id].x - current_position[0],
                        self.customer_by_id[customer_id].y - current_position[1],
                    ),
                )
                ordered.append(next_customer_id)
                customer = self.customer_by_id[next_customer_id]
                current_position = (customer.x, customer.y)
                remaining.remove(next_customer_id)
            optimized_solution["routes"][vehicle_id] = ordered

        optimized_solution["total_distance"] = self._calculate_total_distance(
            optimized_solution["routes"]
        )
        return optimized_solution

    def _improve_service_equity(
        self, solution: Dict[str, Any]
    ) -> Dict[str, Any]:
        """Moderate unusually high or low delivery quantities around their mean."""
        optimized_solution = {
            **solution,
            "routes": {
                vehicle_id: list(customer_ids)
                for vehicle_id, customer_ids in solution["routes"].items()
            },
            "deliveries": dict(solution["deliveries"]),
        }
        deliveries = optimized_solution["deliveries"]
        if not deliveries:
            return optimized_solution

        mean_delivery = float(np.mean(list(deliveries.values())))
        for customer_id, current_delivery in deliveries.items():
            if current_delivery > mean_delivery * 1.2:
                deliveries[customer_id] = mean_delivery * 1.1
            elif current_delivery < mean_delivery * 0.8:
                deliveries[customer_id] = mean_delivery * 0.9
        return optimized_solution

    def _calculate_stakeholder_utilities(
        self, solution: Dict[str, Any]
    ) -> Dict[str, float]:
        """Calculate each stakeholder's weighted, priority-adjusted utility proxy."""
        return {
            stakeholder.name: self._calculate_individual_stakeholder_utility(
                stakeholder, solution
            )
            for stakeholder in self.stakeholders
        }

    def _calculate_individual_stakeholder_utility(
        self,
        stakeholder: Stakeholder,
        solution: Dict[str, Any],
    ) -> float:
        """Score supported metrics and explicitly label unsupported metrics by proxy."""
        utility_score = 0.0
        for metric, weight in stakeholder.utility_function.items():
            if metric == "service_level":
                metric_value = solution.get("service_level", 0.0)
            elif metric == "cost":
                cost_per_customer = solution["total_cost"] / len(self.customers)
                metric_value = np.clip(1.0 - cost_per_customer / 500.0, 0.0, 1.0)
            elif metric in {"environmental_impact", "carbon_emissions"}:
                emissions = (
                    solution["total_distance"] * self.EMISSIONS_PER_DISTANCE_UNIT
                )
                scale = 200.0 if metric == "environmental_impact" else 150.0
                metric_value = np.clip(1.0 - emissions / scale, 0.0, 1.0)
            elif metric == "reliability":
                metric_value = 0.85
            elif metric == "profitability":
                metric_value = np.clip(1.0 - solution["total_cost"] / 1000.0, 0.0, 1.0)
            else:
                # No operational data are available for the remaining utility metrics.
                metric_value = 0.7
            utility_score += float(metric_value) * weight
        return float(np.clip(utility_score * stakeholder.priority, 0.0, 1.0))

    def _finalize_solution(
        self,
        solution: Dict[str, Any],
        compliance_scores: Dict[str, float],
        stakeholder_utilities: Dict[str, float],
    ) -> ValueAlignedSolution:
        """Package final decisions and scores for reporting and period aggregation."""
        environmental_impact = (
            solution["total_distance"] * self.EMISSIONS_PER_DISTANCE_UNIT
        )
        social_impact = (
            1.0 - compliance_scores.get("Social Equity", 0.0)
        ) * 100.0
        overall_ethical_score = float(np.mean(list(compliance_scores.values())))
        return ValueAlignedSolution(
            routes=solution["routes"],
            deliveries=solution["deliveries"],
            total_cost=solution["total_cost"],
            service_level=solution.get("service_level", 0.0),
            ethical_compliance=compliance_scores,
            stakeholder_utility=stakeholder_utilities,
            environmental_impact=environmental_impact,
            social_impact=social_impact,
            economic_impact=solution["total_cost"],
            overall_ethical_score=overall_ethical_score,
        )

    def _report_ethical_results(
        self,
        solution: ValueAlignedSolution,
        compliance_scores: Dict[str, float],
        stakeholder_utilities: Dict[str, float],
    ) -> None:
        """Print the scores with units and make their proxy status explicit."""
        print("\nPeriod results (illustrative proxy metrics):")
        print(f"  Total cost: ${solution.total_cost:.2f}")
        print(f"  Assumed service level: {solution.service_level:.2f}")
        print(f"  Mean principle-compliance score: {solution.overall_ethical_score:.2f}")
        print("  Principle compliance:")
        for principle, score in compliance_scores.items():
            status = "Meets score threshold" if score >= 0.8 else "Below score threshold"
            print(f"    {principle}: {score:.2f} ({status})")
        print("  Stakeholder utility proxies:")
        for stakeholder, utility in stakeholder_utilities.items():
            print(f"    {stakeholder}: {utility:.2f}")
        print(f"  Estimated emissions: {solution.environmental_impact:.1f} kg CO2")
        print(f"  Social-impact proxy: {solution.social_impact:.1f}")
        print(f"  Economic impact (cost): ${solution.economic_impact:.2f}")

    def run_ethical_optimization(self) -> Dict[str, Any]:
        """Evaluate the static instance independently over the requested periods."""
        print(
            f"\nRunning {self.num_periods} independent evaluations "
            "of the static instance."
        )
        for period in range(self.num_periods):
            self.optimize_with_ethical_constraints(period)

        final_results = self._calculate_final_results()
        print("\n=== EVALUATION SUMMARY ===")
        print(f"  Average ethical score: {final_results['avg_ethical_score']:.2f}")
        print(
            "  Average stakeholder utility: "
            f"{final_results['avg_stakeholder_utility']:.2f}"
        )
        print(
            "  Total estimated emissions: "
            f"{final_results['total_environmental_impact']:.1f} kg CO2"
        )
        print(f"  Average cost per evaluation: ${final_results['avg_cost_per_period']:.2f}")
        return final_results

    def _calculate_final_results(self) -> Dict[str, Any]:
        """Aggregate period metrics while retaining their per-period histories."""
        if not self.solution_history:
            return {}

        average_ethical_score = float(
            np.mean([solution.overall_ethical_score for solution in self.solution_history])
        )
        average_stakeholder_utility = float(
            np.mean([
                np.mean(list(solution.stakeholder_utility.values()))
                for solution in self.solution_history
            ])
        )
        total_environmental_impact = float(
            sum(solution.environmental_impact for solution in self.solution_history)
        )
        average_cost_per_period = float(
            np.mean([solution.total_cost for solution in self.solution_history])
        )

        principle_compliance = {}
        for principle in self.principles:
            scores = [
                compliance.get(principle.name, 0.0)
                for compliance in self.compliance_history
            ]
            principle_compliance[principle.name] = {
                "average": float(np.mean(scores)),
                "trend": float(scores[-1] - scores[0]) if len(scores) > 1 else 0.0,
            }

        stakeholder_satisfaction = {}
        for stakeholder in self.stakeholders:
            utilities = [
                utility.get(stakeholder.name, 0.0)
                for utility in self.utility_history
            ]
            stakeholder_satisfaction[stakeholder.name] = {
                "average": float(np.mean(utilities)),
                "trend": (
                    float(utilities[-1] - utilities[0])
                    if len(utilities) > 1
                    else 0.0
                ),
            }

        return {
            "avg_ethical_score": average_ethical_score,
            "avg_stakeholder_utility": average_stakeholder_utility,
            "total_environmental_impact": total_environmental_impact,
            "avg_cost_per_period": average_cost_per_period,
            "principle_compliance": principle_compliance,
            "stakeholder_satisfaction": stakeholder_satisfaction,
            "solution_history": self.solution_history,
            "compliance_history": self.compliance_history,
            "utility_history": self.utility_history,
        }

In [ ]:
def create_ethical_instance():
    """Create a reproducible depot, customer, and vehicle instance."""
    depot = type(
        "Depot",
        (),
        {"x": 0.0, "y": 0.0, "initial_inventory": 1000, "holding_cost": 0.1},
    )()

    np.random.seed(42)
    customers = []
    for index in range(12):
        cluster_center_x = np.random.choice([-15, 0, 15])
        cluster_center_y = np.random.choice([-15, 0, 15])
        x = cluster_center_x + np.random.normal(0, 6)
        y = cluster_center_y + np.random.normal(0, 6)
        customer = type(
            "Customer",
            (),
            {
                "id": index + 1,
                "x": x,
                "y": y,
                "demand_mean": np.random.uniform(12, 28),
                "demand_std": np.random.uniform(3, 6),
                "holding_cost": np.random.uniform(0.1, 0.3),
                "initial_inventory": np.random.uniform(35, 75),
                "min_inventory": np.random.uniform(12, 24),
                "max_inventory": np.random.uniform(85, 125),
            },
        )()
        customers.append(customer)

    vehicles = [
        type(
            "Vehicle",
            (),
            {"id": vehicle_id, "capacity": 120, "fixed_cost": 150, "variable_cost": 2.2},
        )()
        for vehicle_id in range(1, 4)
    ]
    return depot, customers, vehicles


print("Creating the example instance...")
depot, customers, vehicles = create_ethical_instance()

print(f"Depot: ({depot.x:.1f}, {depot.y:.1f}); initial inventory: {depot.initial_inventory}")
print("Customers (first five):")
for customer in customers[:5]:
    print(
        f"  C{customer.id}: ({customer.x:.1f}, {customer.y:.1f}), "
        f"mean demand {customer.demand_mean:.1f} +/- {customer.demand_std:.1f}, "
        f"initial inventory {customer.initial_inventory:.1f}"
    )
print("Vehicles:")
for vehicle in vehicles:
    print(
        f"  V{vehicle.id}: capacity {vehicle.capacity}, "
        f"fixed cost ${vehicle.fixed_cost}, "
        f"variable cost ${vehicle.variable_cost:.2f} per distance unit"
    )

assert len(customers) == 12
assert len(vehicles) == 3
assert len({customer.id for customer in customers}) == len(customers)

In [ ]:
def run_ethical_optimization():
    """Create the framework, run the six-period example, and return its results."""
    ethical_ai = ConstitutionalAI(depot, customers, vehicles, num_periods=6)
    print("\nConstitutional principles:")
    for principle in ethical_ai.principles:
        print(
            f"  {principle.name}: weight {principle.weight:.2f}; "
            f"type {principle.constraint_type}"
        )
    print("\nStakeholders:")
    for stakeholder in ethical_ai.stakeholders:
        print(f"  {stakeholder.name}: priority {stakeholder.priority:.2f}")
    print("\nEthical constraints:")
    for constraint in ethical_ai.ethical_constraints:
        print(
            f"  {constraint.name}: limit {constraint.limit_value}; "
            f"measurement: {constraint.measurement_method}"
        )

    start_time = time.perf_counter()
    results = ethical_ai.run_ethical_optimization()
    elapsed_time = time.perf_counter() - start_time
    print(f"\nEvaluation runtime: {elapsed_time:.2f} seconds")
    return ethical_ai, results


ethical_ai, results = run_ethical_optimization()

# These checks verify that route adjustments preserve customer coverage.
expected_customer_ids = {customer.id for customer in customers}
for solution in results["solution_history"]:
    routed_customer_ids = [
        customer_id
        for route in solution.routes.values()
        for customer_id in route
    ]
    assert set(routed_customer_ids) == expected_customer_ids
    assert len(routed_customer_ids) == len(expected_customer_ids)
    assert set(solution.deliveries) == expected_customer_ids
    assert np.isclose(
        solution.total_cost,
        solution.economic_impact,
        rtol=1e-9,
        atol=1e-9,
    )

In [ ]:
def analyze_ethical_performance(ethical_ai, results):
    """Summarize proxy scores, constraint assessability, and emissions metrics."""
    print("\n=== ETHICAL FRAMEWORK ANALYSIS ===")
    print("\nOverall performance (illustrative metrics):")
    print(f"  Average principle score: {results['avg_ethical_score']:.2f}")
    print(f"  Average stakeholder utility: {results['avg_stakeholder_utility']:.2f}")
    print(
        "  Total estimated emissions: "
        f"{results['total_environmental_impact']:.1f} kg CO2"
    )
    print(f"  Average cost per evaluation: ${results['avg_cost_per_period']:.2f}")

    print("\nPrinciple compliance averages and trends:")
    principle_compliance = results["principle_compliance"]
    for principle_name, data in principle_compliance.items():
        score = data["average"]
        trend = data["trend"]
        status = "Meets score threshold" if score >= 0.8 else "Below score threshold"
        print(f"  {principle_name}: {score:.2f}; change across evaluations {trend:+.2f}; {status}")

    print("\nStakeholder utility averages and trends:")
    stakeholder_satisfaction = results["stakeholder_satisfaction"]
    for stakeholder_name, data in stakeholder_satisfaction.items():
        print(
            f"  {stakeholder_name}: {data['average']:.2f}; "
            f"change across evaluations {data['trend']:+.2f}"
        )

    print("\nConstraint assessment:")
    constraint_satisfaction = {}
    for constraint in ethical_ai.ethical_constraints:
        if constraint.name == "Carbon Emission Limit":
            scores = [
                float(
                    np.clip(
                        1.0
                        - (solution.environmental_impact - constraint.limit_value)
                        / constraint.limit_value,
                        0.0,
                        1.0,
                    )
                )
                for solution in results["solution_history"]
            ]
        elif constraint.name == "Service Equity Requirement":
            scores = [
                ethical_ai._calculate_service_equity(solution.deliveries)
                for solution in results["solution_history"]
            ]
        else:
            print(
                f"  {constraint.name}: Not assessed; "
                f"{constraint.measurement_method}."
            )
            continue
        average_satisfaction = float(np.mean(scores))
        constraint_satisfaction[constraint.name] = average_satisfaction
        print(f"  {constraint.name}: {average_satisfaction:.2f} (proxy score)")

    principle_weights = {principle.name: principle.weight for principle in ethical_ai.principles}
    weighted_compliance = sum(
        data["average"] * principle_weights[name]
        for name, data in principle_compliance.items()
    )
    total_weight = sum(principle_weights.values())
    value_alignment_score = weighted_compliance / total_weight if total_weight else 0.0
    ethical_consistency = float(
        np.std([data["average"] for data in principle_compliance.values()])
    )
    stakeholder_balance = float(
        np.std([data["average"] for data in stakeholder_satisfaction.values()])
    )

    evaluation_count = len(results["solution_history"])
    if evaluation_count == 0:
        raise ValueError("Performance analysis requires at least one evaluation.")
    emissions_per_evaluation = (
        results["total_environmental_impact"] / evaluation_count
    )
    emissions_per_customer = emissions_per_evaluation / len(customers)
    environmental_efficiency = (
        len(customers) / emissions_per_evaluation
        if emissions_per_evaluation > 0
        else float("inf")
    )

    print("\nAggregate proxy indicators:")
    print(f"  Weighted value-alignment score: {value_alignment_score:.2f}")
    print(f"  Principle-score standard deviation: {ethical_consistency:.3f}")
    print(f"  Stakeholder-utility standard deviation: {stakeholder_balance:.3f}")
    print(f"  Estimated emissions per evaluation: {emissions_per_evaluation:.1f} kg CO2")
    print(f"  Estimated emissions per customer: {emissions_per_customer:.2f} kg CO2")
    print(
        "  Customers served per kg CO2 (proxy): "
        f"{environmental_efficiency:.3f}"
    )

    return {
        "value_alignment_score": value_alignment_score,
        "ethical_consistency": ethical_consistency,
        "stakeholder_balance": stakeholder_balance,
        "environmental_efficiency": environmental_efficiency,
        "average_compliance": float(
            np.mean([data["average"] for data in principle_compliance.values()])
        ),
        "average_satisfaction": float(
            np.mean([data["average"] for data in stakeholder_satisfaction.values()])
        ),
        "constraint_satisfaction": constraint_satisfaction,
    }


ethical_analysis = analyze_ethical_performance(ethical_ai, results)

In [ ]:
def visualize_ethical_results(ethical_ai, results, ethical_analysis):
    """Plot the simulated compliance, utilities, impacts, and proxy indicators."""
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle("Value-Aligned Framework: Simulated Metrics", fontsize=16, fontweight="bold")

    periods = np.arange(1, len(results["compliance_history"]) + 1)

    ax = axes[0, 0]
    for principle in ethical_ai.principles[:3]:
        scores = [
            compliance.get(principle.name, 0.0)
            for compliance in results["compliance_history"]
        ]
        ax.plot(periods, scores, marker="o", linewidth=2, label=principle.name)
    ax.set_title("Principle Scores by Evaluation")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Proxy score")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[0, 1]
    stakeholder_names = list(results["stakeholder_satisfaction"])
    average_utilities = [
        results["stakeholder_satisfaction"][name]["average"]
        for name in stakeholder_names
    ]
    ax.bar(stakeholder_names, average_utilities, alpha=0.8)
    ax.set_title("Stakeholder Utility Proxies")
    ax.set_ylabel("Average utility")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[0, 2]
    emissions = [
        solution.environmental_impact for solution in results["solution_history"]
    ]
    ax.plot(periods, emissions, color="tab:red", marker="o", linewidth=2)
    ax.axhline(
        y=ethical_ai.principles[0].target_value,
        color="tab:green",
        linestyle="--",
        label="Principle target",
    )
    ax.set_title("Estimated Emissions")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Estimated emissions (kg CO2)")
    ax.grid(True, alpha=0.3)
    ax.legend()

    ax = axes[1, 0]
    principle_names = list(results["principle_compliance"])
    average_scores = [
        results["principle_compliance"][name]["average"]
        for name in principle_names
    ]
    ax.barh(principle_names, average_scores, alpha=0.8)
    ax.set_title("Average Principle Scores")
    ax.set_xlabel("Average proxy score")
    ax.set_xlim(0, 1)
    ax.grid(True, axis="x", alpha=0.3)

    ax = axes[1, 1]
    costs = [solution.total_cost for solution in results["solution_history"]]
    ethical_scores = [
        solution.overall_ethical_score for solution in results["solution_history"]
    ]
    scatter = ax.scatter(costs, ethical_scores, c=periods, cmap="viridis", s=80)
    ax.set_title("Cost and Mean Principle Score")
    ax.set_xlabel("Total cost ($)")
    ax.set_ylabel("Mean principle score")
    ax.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax)
    colorbar.set_label("Evaluation")

    ax = axes[1, 2]
    utility_history = results["utility_history"]
    for stakeholder_name in stakeholder_names:
        values = [
            utilities.get(stakeholder_name, 0.0) for utilities in utility_history
        ]
        ax.plot(periods, values, marker="o", label=stakeholder_name)
    ax.set_title("Stakeholder Utility Proxies by Evaluation")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Utility proxy")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    ax = axes[2, 0]
    assessed_constraints = list(ethical_analysis["constraint_satisfaction"])
    if assessed_constraints:
        matrix = np.asarray(
            [
                [
                    ethical_analysis["constraint_satisfaction"][name]
                    for _ in periods
                ]
                for name in assessed_constraints
            ]
        )
        image = ax.imshow(matrix, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)
        ax.set_yticks(np.arange(len(assessed_constraints)))
        ax.set_yticklabels(assessed_constraints)
        ax.set_xticks(np.arange(len(periods)))
        ax.set_xticklabels(periods)
        colorbar = fig.colorbar(image, ax=ax)
        colorbar.set_label("Proxy satisfaction score")
    else:
        ax.text(0.5, 0.5, "No constraints assessed", ha="center", va="center")
    ax.set_title("Assessed Constraint Proxies")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Constraint")

    ax = axes[2, 1]
    value_alignment = ethical_analysis["value_alignment_score"]
    ax.plot(periods, [value_alignment] * len(periods), color="tab:purple", marker="o")
    ax.set_title("Weighted Value-Alignment Proxy")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Weighted score")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    ax = axes[2, 2]
    ethical_efficiency = [
        solution.overall_ethical_score for solution in results["solution_history"]
    ]
    economic_efficiency = [
        float(np.clip(1.0 - solution.total_cost / 2000.0, 0.0, 1.0))
        for solution in results["solution_history"]
    ]
    x_positions = np.arange(len(periods))
    width = 0.35
    ax.bar(
        x_positions - width / 2,
        ethical_efficiency,
        width,
        label="Mean principle score",
        alpha=0.8,
    )
    ax.bar(
        x_positions + width / 2,
        economic_efficiency,
        width,
        label="Normalized cost proxy",
        alpha=0.8,
    )
    ax.set_title("Two Separately Defined Proxies")
    ax.set_xlabel("Evaluation")
    ax.set_ylabel("Score")
    ax.set_xticks(x_positions)
    ax.set_xticklabels(periods)
    ax.set_ylim(0, 1)
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(fontsize=8)

    fig.tight_layout()
    plt.show()


visualize_ethical_results(ethical_ai, results, ethical_analysis)

In [ ]:
def report_value_aligned_metrics(ethical_ai, results, ethical_analysis):
    """Present the modeled indicators without treating illustrative values as benchmarks."""
    print("\n=== VALUE-ALIGNED MODEL INDICATORS ===")
    print("These are outputs of the stated toy model, not comparative benchmark results.")

    metric_rows = [
        ("Average principle score", results["avg_ethical_score"], "unitless proxy"),
        (
            "Weighted value-alignment score",
            ethical_analysis["value_alignment_score"],
            "unitless proxy",
        ),
        (
            "Average stakeholder utility",
            results["avg_stakeholder_utility"],
            "unitless proxy",
        ),
        (
            "Average cost per evaluation",
            results["avg_cost_per_period"],
            "cost units",
        ),
        (
            "Total estimated emissions",
            results["total_environmental_impact"],
            "kg CO2 across evaluations",
        ),
    ]
    print(f"\n{'Indicator':<38} {'Value':>12}  Unit or interpretation")
    print("-" * 82)
    for name, value, unit in metric_rows:
        print(f"{name:<38} {value:>12.3f}  {unit}")

    print("\nAssessment coverage:")
    assessed_names = set(ethical_analysis["constraint_satisfaction"])
    for constraint in ethical_ai.ethical_constraints:
        status = "assessed with a proxy" if constraint.name in assessed_names else "not assessed"
        print(f"  {constraint.name}: {status}")

    print("\nInterpretation:")
    print(
        "  Costs and emissions are calculated from the stated distance and vehicle-cost assumptions."
    )
    print(
        "  Principle and stakeholder values are constructed scores; they are not observed outcomes."
    )
    print(
        "  A constraint marked not assessed cannot be considered satisfied from this simulation."
    )

    return {
        "average_principle_score": results["avg_ethical_score"],
        "weighted_value_alignment_proxy": ethical_analysis["value_alignment_score"],
        "average_stakeholder_utility_proxy": results["avg_stakeholder_utility"],
        "average_cost_per_evaluation": results["avg_cost_per_period"],
        "total_estimated_emissions": results["total_environmental_impact"],
        "assessed_constraints": sorted(assessed_names),
    }


value_aligned_metrics = report_value_aligned_metrics(
    ethical_ai, results, ethical_analysis
)

## Interpreting the Example

### What the scores mean

- **Principle-compliance scores** are normalized proxies defined in `_calculate_principle_compliance`. Their arithmetic mean is reported as the overall ethical score. The weighted value-alignment indicator in the analysis uses the principle weights instead; these two summaries answer different aggregation questions.
- **Stakeholder utilities** combine the stakeholder's specified metric weights and priority. Unsupported metrics use an explicitly simplified proxy, so a utility score should not be read as a measured level of satisfaction.
- **Estimated emissions** equal total Euclidean route distance multiplied by the assumed emissions factor. They are not lifecycle emissions or a vehicle-specific emissions estimate.
- **Constraint penalties** are included only for carbon emissions and service equity, the two constraints with usable measures in this instance. The remaining listed constraints are reported as not assessed rather than being assigned invented satisfaction scores.
- **The six evaluations** repeat the same static data and deterministic rules. Identical period results therefore indicate repetition of one instance, not learning, adaptation, or a time trend in operations.

### Limitations and possible extensions

This example does not model inventory balance, replenishment timing, vehicle-capacity feasibility, uncertain demand, labor hours, resource consumption, or measured community impacts. Consequently, it cannot establish feasibility or ethical compliance for a complete inventory-routing problem. Any extension should first provide defensible operational data and units for the additional measures, then state how they enter the existing staged evaluation. The route-ordering heuristic is not an optimality guarantee, and the illustrative scores depend on the chosen assumptions and normalizations.

A responsible interpretation separates three questions: what the model calculates, what the proxy is intended to represent, and what evidence would be needed to support a real-world ethical claim. This distinction is essential when reporting value-aligned optimization results.